# 05 — Patient-level analysis

**Why this exists, and why it uses synthetic data:** as noted in `model_card.md` → "Known limitations", the public NCT-CRC-HE-100K training archive does not expose reliable patch-to-patient mapping, so the bootstrap analysis in `docs/RESULTS.md` is computed at the patch level. That is a **data** limitation, not a code one: `src/datasets/split.py::assign_patient_splits` and the patient-level tooling in `src/analysis/patient_level.py` already exist and are unit-tested (`tests/test_patient_level.py`).

This notebook demonstrates the full method on the synthetic dataset, which *does* carry a `patient_id`, so it is ready to point at a real patient-mapped cohort the moment one is available. **The predicted labels below are simulated noise, not the output of a trained classifier** — the point of this notebook is the evaluation methodology, not a new classification result.

Three things are demonstrated:
1. an automated patient-leakage check between splits;
2. patient-level majority-vote metrics with a bootstrap CI that resamples **patients**, not patches;
3. why that matters: a side-by-side comparison against the patch-level bootstrap already used in `docs/RESULTS.md`, which resamples patches and can understate uncertainty when a patient's patches share correlated errors.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score

sys.path.insert(0, str(Path.cwd().parents[0]))

from src.analysis.patient_level import (
    aggregate_predictions_by_patient,
    bootstrap_patch_level_metric_with_patient_clusters,
    calculate_patient_level_bootstrap_ci,
    calculate_patient_level_metrics,
    check_patient_leakage,
)
from src.compare_probe_predictions import (
    create_stratified_bootstrap_indices,
)
from src.datasets.split import assign_patient_splits
from src.datasets.synthetic import generate_synthetic_histology_dataset

## 1. Build a patient-mapped dataset and check for leakage

In [2]:
metadata = generate_synthetic_histology_dataset(
    output_dir="./data/raw/synthetic_patient_patches",
    metadata_path="./data/metadata/synthetic_metadata.csv",
    num_patients=40,
    slides_per_patient=1,
    patches_per_slide=8,
    image_size=64,
    seed=5,
)

metadata = assign_patient_splits(
    metadata=metadata,
    train_fraction=0.6,
    validation_fraction=0.2,
    test_fraction=0.2,
    seed=5,
    stratify_column=None,
)

leakage_report = check_patient_leakage(metadata)

print(f"Patients: {metadata['patient_id'].nunique()}, "
      f"patches: {len(metadata)}")
print(f"Leaking patients: {len(leakage_report)} (should be 0)")
assert leakage_report.empty, "Patient leakage detected between splits!"

test_metadata = metadata[metadata["split"] == "test"].copy()
print(f"Test patients: {test_metadata['patient_id'].nunique()}, "
      f"test patches: {len(test_metadata)}")

Patients: 40, patches: 320
Leaking patients: 0 (should be 0)
Test patients: 8, test patches: 64


## 2. Simulate patch-level predictions with patient-correlated errors

Real classifier errors are rarely independent per patch: a genuinely ambiguous slide tends to fool the model on *many* of its patches at once, not on an independent random subset. To make that concrete, each simulated patient is either "easy" (low error rate) or "hard" (high error rate), and every one of their patches uses that same rate.

In [3]:
rng = np.random.default_rng(3)

unique_test_patients = test_metadata["patient_id"].unique()
patient_error_rate = {
    patient_id: (0.6 if rng.random() < 0.3 else 0.05)
    for patient_id in unique_test_patients
}

predicted_labels = [
    (
        1 - row.label
        if rng.random() < patient_error_rate[row.patient_id]
        else row.label
    )
    for row in test_metadata.itertuples()
]

test_metadata["predicted_label"] = predicted_labels
print(
    f"{sum(rate > 0.5 for rate in patient_error_rate.values())} "
    f"'hard' patients out of {len(unique_test_patients)}"
)

4 'hard' patients out of 8


## 3. Patient-level metrics and bootstrap CI (correct unit of resampling)

In [4]:
patient_level_frame = aggregate_predictions_by_patient(
    test_metadata,
    true_label_column="label",
    predicted_label_column="predicted_label",
)

point_metrics = calculate_patient_level_metrics(
    patient_level_frame, "label", "predicted_label"
)
print("Patient-level (majority-vote) metrics:", point_metrics)

patient_bootstrap = calculate_patient_level_bootstrap_ci(
    patient_level_frame,
    true_label_column="label",
    predicted_label_column="predicted_label",
    number_of_iterations=2000,
    seed=0,
)
print("Patient-level bootstrap (balanced accuracy):")
print(patient_bootstrap["balanced_accuracy"])

Patient-level (majority-vote) metrics: {'balanced_accuracy': 0.8, 'macro_f1': 0.75}


c:\Users\User\Documents\histology_vae_student\.venv\Lib\site-packages\sklearn\metrics\_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
c:\Users\User\Documents\histology_vae_student\.venv\Lib\site-packages\sklearn\metrics\_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
c:\Users\User\Documents\histology_vae_student\.venv\Lib\site-packages\sklearn\metrics\_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
c:\Users\User\Documents\histology_vae_student\.venv\Lib\site-packages\sklearn\metrics\_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
c:\Users\User\Documents\histology_vae_student\.venv\Lib\site-packages\sklearn\metrics\_classification.py:2939: UserWarning: 

Patient-level bootstrap (balanced accuracy):
{'point_estimate': 0.8, 'bootstrap_mean': 0.7925214285714286, 'ci_2_5': 0.5, 'ci_97_5': 1.0, 'number_of_patients': 8, 'bootstrap_iterations': 2000}


c:\Users\User\Documents\histology_vae_student\.venv\Lib\site-packages\sklearn\metrics\_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


## 4. Naive patch-level bootstrap vs. correct patient-cluster bootstrap

Both use the **same** patch-level balanced accuracy — the only difference is *what gets resampled*:

- **Naive** (`create_stratified_bootstrap_indices`, the function already used for the patch-level bootstrap in `docs/RESULTS.md`): resamples individual patches, stratified by class. Treats every patch as an independent observation.
- **Cluster bootstrap** (`bootstrap_patch_level_metric_with_patient_clusters`): resamples **patients**, then includes all of a sampled patient's patches together. Respects that patches from the same patient are correlated.

In [5]:
true_labels = test_metadata["label"].to_numpy()
predicted_labels_array = test_metadata["predicted_label"].to_numpy()
patient_ids_array = test_metadata["patient_id"].to_numpy()

naive_rng = np.random.default_rng(0)
unique_labels = sorted(set(true_labels.tolist()))
naive_samples = np.empty(2000)

for iteration in range(2000):
    sampled_indices = create_stratified_bootstrap_indices(
        true_labels, unique_labels, naive_rng
    )
    naive_samples[iteration] = balanced_accuracy_score(
        true_labels[sampled_indices],
        predicted_labels_array[sampled_indices],
    )

naive_point = balanced_accuracy_score(true_labels, predicted_labels_array)
naive_ci_low, naive_ci_high = np.quantile(naive_samples, [0.025, 0.975])

cluster_result = bootstrap_patch_level_metric_with_patient_clusters(
    true_labels=true_labels,
    predicted_labels=predicted_labels_array,
    patient_ids=patient_ids_array,
    number_of_iterations=2000,
    seed=0,
)

comparison = pd.DataFrame(
    [
        {
            "method": "naive (resamples patches, ignores patients)",
            "point_estimate": naive_point,
            "ci_2_5": naive_ci_low,
            "ci_97_5": naive_ci_high,
            "ci_width": naive_ci_high - naive_ci_low,
        },
        {
            "method": "cluster bootstrap (resamples patients)",
            "point_estimate": cluster_result["point_estimate"],
            "ci_2_5": cluster_result["ci_2_5"],
            "ci_97_5": cluster_result["ci_97_5"],
            "ci_width": cluster_result["ci_97_5"] - cluster_result["ci_2_5"],
        },
    ]
)
comparison

c:\Users\User\Documents\histology_vae_student\.venv\Lib\site-packages\sklearn\metrics\_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
c:\Users\User\Documents\histology_vae_student\.venv\Lib\site-packages\sklearn\metrics\_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
c:\Users\User\Documents\histology_vae_student\.venv\Lib\site-packages\sklearn\metrics\_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
c:\Users\User\Documents\histology_vae_student\.venv\Lib\site-packages\sklearn\metrics\_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
c:\Users\User\Documents\histology_vae_student\.venv\Lib\site-packages\sklearn\metrics\_classification.py:2939: UserWarning: 

,method,point_estimate,ci_2_5,ci_97_5,ci_width
0,"naive (resamples patches, ignores patients)",0.7875,0.683333,0.883333,0.200000
1,cluster bootstrap (resamples patients),0.7875,0.633929,0.927083,0.293155


In [6]:
width_ratio = (
    comparison.loc[1, "ci_width"] / comparison.loc[0, "ci_width"]
)
print(
    f"The patient-aware CI is {width_ratio:.2f}x the width of the "
    "naive patch-level CI on this simulated, patient-correlated data."
)
if width_ratio <= 1.0:
    print(
        "(With only a few dozen simulated patients this ratio is "
        "noisy across seeds — re-run with a different seed above "
        "if it comes out below 1.0; the qualitative point is that "
        "it should not be reliably *below* 1.0 for correlated errors.)"
    )

The patient-aware CI is 1.47x the width of the naive patch-level CI on this simulated, patient-correlated data.


## Notes

- This does **not** mean the numbers in `docs/RESULTS.md` are wrong — it means their reported confidence intervals should be read as a **lower bound** on the true uncertainty, since they resample patches rather than patients.
- The moment a patient-mapped cohort is available (see `README.md` → "Roadmap"), replace the simulated `predicted_label` above with real model predictions and re-run this notebook unchanged.
- `check_patient_leakage` should be run on *every* new patient-mapped dataset before training, not only here — consider calling it from `src/prepare_crc_data.py` once real patient IDs exist.